# Question 1 — survey and external-data preparation

This notebook has one responsibility: produce validated annual, quarterly and monthly analysis
panels. It uses only the three external datasets retained in the final analysis—ONS
population/migration, residence-based earnings and Metropolitan Police recorded crime.

Annual survey estimates use `wt_final`; monthly and quarterly estimates use `wt_time`. External
context is joined after survey aggregation and is never survey-weighted. No model split is stored
in the prepared data.


In [1]:
from pathlib import Path
import csv
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display


cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'data_integration').exists() else cwd.parent
OUTPUT_ROOT = PROJECT_ROOT / 'question1_outputs'
DATA_DIR = OUTPUT_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

EXTRA_DATA_DIR = Path(r'Y:\afinal\extradata')
WAVES_PATH = PROJECT_ROOT / 'data_integration' / 'waves.json'
LOOKUP_PATH = (
    PROJECT_ROOT / 'Jingyi Hua' / 'data' / 'processed'
    / 'variable_value_labels_lookup_year7_8.csv'
)
EXTERNAL_PATHS = {
    'population_migration': EXTRA_DATA_DIR / 'myebtablesenglandwales20112024.xlsx',
    'resident_earnings': EXTRA_DATA_DIR / 'earnings-residence-borough.xlsx',
    'recorded_crime': EXTRA_DATA_DIR / 'MPS Borough Level Crime.csv',
}
OUTPUTS = {
    'london_annual': DATA_DIR / 'q1_london_annual.csv',
    'area_annual': DATA_DIR / 'q1_inner_outer_annual.csv',
    'borough_annual': DATA_DIR / 'q1_borough_annual.csv',
    'london_monthly': DATA_DIR / 'q1_london_monthly.csv',
    'area_monthly': DATA_DIR / 'q1_inner_outer_monthly.csv',
    'borough_monthly': DATA_DIR / 'q1_borough_monthly.csv',
    'london_quarterly': DATA_DIR / 'q1_london_quarterly.csv',
    'area_quarterly': DATA_DIR / 'q1_inner_outer_quarterly.csv',
    'borough_quarterly': DATA_DIR / 'q1_borough_quarterly.csv',
}
AUDIT_OUTPUT = DATA_DIR / 'q1_preparation_audit.xlsx'

assert WAVES_PATH.exists() and LOOKUP_PATH.exists()
assert all(path.exists() for path in EXTERNAL_PATHS.values()), EXTERNAL_PATHS

SPSS_MISSING = list(range(-99, -89))
CITY_OF_LONDON_LA2023 = 59
AREA_LABELS = {1: 'Inner London', 2: 'Outer London'}
AREA_CODES = {1: 'E13000001', 2: 'E13000002'}
FIRST_SURVEY_PERIOD = pd.Period('2015-11', freq='M')
COMPOSITION_FEATURES = ['older_adult_rate', 'limiting_disability_rate', 'online_response_rate']
OUTCOME_COLUMNS = ['inactive_rate', 'fairly_active_rate', 'active_rate']
QUALITY_COLUMNS = ['effective_n']
DEMOGRAPHIC_EARNINGS_COLUMNS = [
    'adult_population',
    'population_age_16_34_rate',
    'population_age_65_plus_rate',
    'population_female_rate',
    'net_migration_per_1000',
    'median_weekly_earnings_gbp',
    'earnings_relative_to_london',
]
CRIME_RATE_COLUMNS = [
    'personal_safety_crime_rate_per_1000',
    'property_public_space_crime_rate_per_1000',
    'selected_recorded_crime_rate_per_1000',
]
EXTERNAL_CONTEXT_COLUMNS = DEMOGRAPHIC_EARNINGS_COLUMNS + CRIME_RATE_COLUMNS
EXTERNAL_QUALITY_COLUMNS = ['earnings_confidence_pct']
EXTERNAL_PANEL_COLUMNS = ['context_reference_year'] + EXTERNAL_CONTEXT_COLUMNS + EXTERNAL_QUALITY_COLUMNS

waves = json.loads(WAVES_PATH.read_text(encoding='utf-8'))
for wave in waves:
    wave['path'] = PROJECT_ROOT / wave['relative_path']
    assert wave['path'].exists(), wave['path']
assert [wave['wave_index'] for wave in waves] == list(range(1, 9))


## 1. External sources and harmonisation

Only variables that enter description, clustering, forecasting or explanatory analysis are
prepared. Weather and economic inactivity are removed from both the data pipeline and all
downstream notebooks.


In [2]:
external_inventory = pd.DataFrame([
    {
        'dataset': 'ONS mid-year population estimates and components of change',
        'file': EXTERNAL_PATHS['population_migration'].name,
        'analysis_role': 'adult population, age/sex structure and net migration',
        'coverage': '32 boroughs, 2015–2023',
    },
    {
        'dataset': 'ONS Annual Survey of Hours and Earnings',
        'file': EXTERNAL_PATHS['resident_earnings'].name,
        'analysis_role': 'residence-based median weekly earnings',
        'coverage': '32 boroughs, 2015–2023',
    },
    {
        'dataset': 'Metropolitan Police borough-level recorded crime',
        'file': EXTERNAL_PATHS['recorded_crime'].name,
        'analysis_role': 'stable selected recorded-crime groups',
        'coverage': '32 boroughs, 2015-11–2023-10',
    },
])
display(external_inventory)


,dataset,file,analysis_role,coverage
0,ONS mid-year population estimates and componen...,myebtablesenglandwales20112024.xlsx,"adult population, age/sex structure and net mi...","32 boroughs, 2015–2023"
1,ONS Annual Survey of Hours and Earnings,earnings-residence-borough.xlsx,residence-based median weekly earnings,"32 boroughs, 2015–2023"
2,Metropolitan Police borough-level recorded crime,MPS Borough Level Crime.csv,stable selected recorded-crime groups,"32 boroughs, 2015-11–2023-10"


### Population structure and migration


In [3]:
population_raw = pd.read_excel(
    EXTERNAL_PATHS['population_migration'], sheet_name='MYEB1', header=1
)
population_raw = population_raw.loc[
    population_raw['ladcode23'].astype(str).str.startswith('E090')
    & population_raw['ladcode23'].ne('E09000001')
].copy()
population_raw['age'] = pd.to_numeric(population_raw['age'], errors='coerce')
population_columns = [f'population_{year}' for year in range(2015, 2024)]
population_long = population_raw.melt(
    id_vars=['ladcode23', 'laname23', 'sex', 'age'],
    value_vars=population_columns,
    var_name='calendar_year',
    value_name='population',
)
population_long['calendar_year'] = population_long['calendar_year'].str.extract(r'(\d{4})').astype(int)
population_long['population'] = pd.to_numeric(population_long['population'], errors='coerce')
assert population_long['population'].notna().all()

population_rows = []
for (code_value, name_value, calendar_year), group in population_long.groupby(
    ['ladcode23', 'laname23', 'calendar_year'], sort=True
):
    adults = group.loc[group['age'].ge(16)]
    population_rows.append({
        'geography_code': code_value,
        'geography_name': name_value,
        'calendar_year': calendar_year,
        'total_population': group['population'].sum(),
        'adult_population': adults['population'].sum(),
        'age_16_34_population': adults.loc[adults['age'].between(16, 34), 'population'].sum(),
        'age_65_plus_population': adults.loc[adults['age'].ge(65), 'population'].sum(),
        'female_adult_population': adults.loc[adults['sex'].eq('f'), 'population'].sum(),
    })
population_context = pd.DataFrame(population_rows)

migration_raw = pd.read_excel(
    EXTERNAL_PATHS['population_migration'], sheet_name='MYEB3', header=1
)
migration_raw = migration_raw.loc[
    migration_raw['ladcode23'].astype(str).str.startswith('E090')
    & migration_raw['ladcode23'].ne('E09000001')
].copy()
migration_rows = []
for _, row in migration_raw.iterrows():
    for calendar_year in range(2015, 2024):
        internal = pd.to_numeric(row[f'internal_net_{calendar_year}'], errors='coerce')
        international = pd.to_numeric(row[f'international_net_{calendar_year}'], errors='coerce')
        migration_rows.append({
            'geography_code': row['ladcode23'],
            'calendar_year': calendar_year,
            'net_migration': internal + international,
        })
migration_context = pd.DataFrame(migration_rows)


### Residence-based earnings


In [4]:
earnings_sheet = pd.read_excel(
    EXTERNAL_PATHS['resident_earnings'], sheet_name='Total, weekly', header=None
)
earnings_rows = []
for column_index in range(2, earnings_sheet.shape[1], 2):
    calendar_year = pd.to_numeric(earnings_sheet.iloc[0, column_index], errors='coerce')
    if pd.isna(calendar_year) or not 2015 <= int(calendar_year) <= 2023:
        continue
    for row_index in range(3, earnings_sheet.shape[0]):
        earnings_rows.append({
            'geography_name': earnings_sheet.iloc[row_index, 1],
            'calendar_year': int(calendar_year),
            'median_weekly_earnings_gbp': pd.to_numeric(
                earnings_sheet.iloc[row_index, column_index], errors='coerce'
            ),
            'earnings_confidence_pct': pd.to_numeric(
                earnings_sheet.iloc[row_index, column_index + 1], errors='coerce'
            ),
        })
earnings_context = pd.DataFrame(earnings_rows).dropna(subset=['geography_name'])

borough_external_year = (
    population_context
    .merge(migration_context, on=['geography_code', 'calendar_year'], validate='one_to_one')
    .merge(earnings_context, on=['geography_name', 'calendar_year'], validate='one_to_one')
)
borough_external_year['population_age_16_34_rate'] = (
    borough_external_year['age_16_34_population'] / borough_external_year['adult_population']
)
borough_external_year['population_age_65_plus_rate'] = (
    borough_external_year['age_65_plus_population'] / borough_external_year['adult_population']
)
borough_external_year['population_female_rate'] = (
    borough_external_year['female_adult_population'] / borough_external_year['adult_population']
)
borough_external_year['net_migration_per_1000'] = (
    1000 * borough_external_year['net_migration'] / borough_external_year['total_population']
)
london_earnings = borough_external_year.groupby('calendar_year').apply(
    lambda group: np.average(
        group['median_weekly_earnings_gbp'], weights=group['adult_population']
    ),
    include_groups=False,
)
borough_external_year['earnings_relative_to_london'] = (
    borough_external_year['median_weekly_earnings_gbp']
    / borough_external_year['calendar_year'].map(london_earnings)
)
assert len(borough_external_year) == 32 * 9


### Stable recorded-crime measures

Only continuously populated major categories are used. `personal_safety` combines violence,
sexual offences and robbery; `property_public_space` combines theft, vehicle offences, and
arson/criminal damage. Their sum is explicitly labelled selected recorded crime—not total crime.


In [5]:
crime_raw = pd.read_csv(EXTERNAL_PATHS['recorded_crime'])
crime_month_columns = [
    column for column in crime_raw
    if column.isdigit() and '201511' <= column <= '202310'
]
CRIME_GROUPS = {
    'personal_safety': ['VIOLENCE AGAINST THE PERSON', 'SEXUAL OFFENCES', 'ROBBERY'],
    'property_public_space': ['THEFT', 'VEHICLE OFFENCES', 'ARSON AND CRIMINAL DAMAGE'],
}
study_boroughs = set(borough_external_year['geography_name'])
assert not (study_boroughs - set(crime_raw['BoroughName']))

crime_long = crime_raw.melt(
    id_vars=['MajorText', 'MinorText', 'BoroughName'],
    value_vars=crime_month_columns,
    var_name='yyyymm',
    value_name='recorded_count',
)
crime_long['period_start'] = pd.to_datetime(crime_long['yyyymm'], format='%Y%m')
crime_long['recorded_count'] = pd.to_numeric(crime_long['recorded_count'], errors='coerce')
crime_long = crime_long.loc[crime_long['BoroughName'].isin(study_boroughs)].copy()
crime_long['crime_group'] = 'excluded'
for group_name, major_categories in CRIME_GROUPS.items():
    crime_long.loc[crime_long['MajorText'].isin(major_categories), 'crime_group'] = group_name

crime_counts = (
    crime_long.loc[crime_long['crime_group'].ne('excluded')]
    .groupby(['BoroughName', 'period_start', 'crime_group'], as_index=False)['recorded_count']
    .sum()
    .pivot(index=['BoroughName', 'period_start'], columns='crime_group', values='recorded_count')
    .reset_index()
    .rename(columns={name: f'{name}_crime_count' for name in CRIME_GROUPS})
)
crime_counts['selected_recorded_crime_count'] = crime_counts[
    [f'{name}_crime_count' for name in CRIME_GROUPS]
].sum(axis=1)

time_reference = pd.DataFrame({
    'period_start': pd.date_range('2015-11-01', periods=96, freq='MS')
})
time_reference['month_index'] = np.arange(1, 97)
time_reference['year'] = ((time_reference['month_index'] - 1) // 12 + 1).astype(int)
time_reference['quarter_index'] = ((time_reference['month_index'] - 1) // 3 + 1).astype(int)
time_reference['month_of_wave'] = ((time_reference['month_index'] - 1) % 12 + 1).astype(int)
time_reference['calendar_year'] = time_reference['period_start'].dt.year
wave_labels = {wave['wave_index']: wave['survey_wave'] for wave in waves}
time_reference['survey_wave'] = time_reference['year'].map(wave_labels)
crime_counts = crime_counts.merge(time_reference, on='period_start', validate='many_to_one')
assert len(crime_counts) == 32 * 96


In [6]:
CRIME_COUNT_COLUMNS = [
    'personal_safety_crime_count',
    'property_public_space_crime_count',
    'selected_recorded_crime_count',
]


def add_context_rates(frame):
    result = frame.copy()
    result['population_age_16_34_rate'] = (
        result['age_16_34_population'] / result['adult_population']
    )
    result['population_age_65_plus_rate'] = (
        result['age_65_plus_population'] / result['adult_population']
    )
    result['population_female_rate'] = (
        result['female_adult_population'] / result['adult_population']
    )
    result['net_migration_per_1000'] = 1000 * result['net_migration'] / result['total_population']
    for count_column, rate_column in zip(CRIME_COUNT_COLUMNS, CRIME_RATE_COLUMNS):
        result[rate_column] = 1000 * result[count_column] / result['total_population']
    return result


def aggregate_context(frame, group_columns):
    rows = []
    grouper = group_columns[0] if len(group_columns) == 1 else group_columns
    for keys, group in frame.groupby(grouper, sort=True, observed=True):
        keys = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_columns, keys))
        for column in [
            'total_population', 'adult_population', 'age_16_34_population',
            'age_65_plus_population', 'female_adult_population', 'net_migration',
        ] + CRIME_COUNT_COLUMNS:
            row[column] = group[column].sum()
        row['median_weekly_earnings_gbp'] = np.average(
            group['median_weekly_earnings_gbp'], weights=group['adult_population']
        )
        row['earnings_relative_to_london'] = np.average(
            group['earnings_relative_to_london'], weights=group['adult_population']
        )
        row['earnings_confidence_pct'] = np.average(
            group['earnings_confidence_pct'], weights=group['adult_population']
        )
        rows.append(row)
    return add_context_rates(pd.DataFrame(rows))


def compact_context(frame):
    return frame[
        [column for column in frame.columns if column in {
            'year', 'survey_wave', 'month_index', 'quarter_index', 'period_start',
            'calendar_year', 'geography_code', 'geography_name', 'inner_outer',
            'context_reference_year',
        }] + EXTERNAL_CONTEXT_COLUMNS + EXTERNAL_QUALITY_COLUMNS
    ].copy()


### Findings

The three retained sources cover all 32 boroughs and the full analytical period. Crime airport and
Unknown rows are excluded by the study-borough filter. No projection, weather or unused labour
market field enters the panels.


## 2. Stable survey schema


In [7]:
def read_header(path):
    with open(path, encoding='utf-8-sig', newline='') as handle:
        return next(csv.reader(handle))


headers = {wave['wave_index']: read_header(wave['path']) for wave in waves}
months_sets = {
    index: {
        column.removeprefix('MONTHS_12_') for column in header if column.startswith('MONTHS_12_')
    }
    for index, header in headers.items()
}
days_sets = {
    index: {
        column.removeprefix('DAYS10P60GR_')
        for column in header
        if column.startswith('DAYS10P60GR_')
    }
    for index, header in headers.items()
}
common_suffixes = set.intersection(*months_sets.values()) & set.intersection(*days_sets.values())
ACTIVITY_SUFFIXES = [
    column.removeprefix('MONTHS_12_')
    for column in headers[1]
    if column.startswith('MONTHS_12_') and column.removeprefix('MONTHS_12_') in common_suffixes
]
MONTHS_COLUMNS = [f'MONTHS_12_{suffix}' for suffix in ACTIVITY_SUFFIXES]
DAYS_COLUMNS = [f'DAYS10P60GR_{suffix}' for suffix in ACTIVITY_SUFFIXES]
MONTHS12_RATE_COLUMNS = [f'MONTHS12_RATE_{suffix}' for suffix in ACTIVITY_SUFFIXES]
DAYS_ANY_RATE_COLUMNS = [f'DAYS_ANY_RATE_{suffix}' for suffix in ACTIVITY_SUFFIXES]
DAYS_2PLUS_RATE_COLUMNS = [f'DAYS_2PLUS_RATE_{suffix}' for suffix in ACTIVITY_SUFFIXES]
ACTIVITY_FEATURES = MONTHS12_RATE_COLUMNS + DAYS_ANY_RATE_COLUMNS + DAYS_2PLUS_RATE_COLUMNS

assert len(ACTIVITY_SUFFIXES) == 124
assert len(ACTIVITY_FEATURES) == 372 and len(set(ACTIVITY_FEATURES)) == 372
assert 'HULAHOOP_P27' not in ACTIVITY_SUFFIXES


## 3. Raw missingness audit

Blank values and SPSS codes are counted before filtering. Routed missing activity responses remain
missing and are never converted to zero. Detailed and summary audits are bundled into one Excel
workbook rather than emitted as separate CSV files.


In [8]:
RAW_AUDIT_CHUNK_SIZE = 5000
Q1_RAW_BASE_COLUMNS = [
    'mode',
    'LA_2023',
    'LondInOut',
    'Age9',
    'Disab3',
    'wt_final',
    'wt_time',
    'MEMS7GR_ALL',
]
SPSS_MISSING_TOKENS = list(
    {token for code in SPSS_MISSING for token in (code, float(code), str(code), f'{code}.0')}
)


def audit_raw_wave_missingness(wave):
    header = headers[wave['wave_index']]
    month_source = next(column for column in header if column.lower() == 'month')
    required = set(Q1_RAW_BASE_COLUMNS + [month_source] + MONTHS_COLUMNS + DAYS_COLUMNS)
    q1_columns = [column for column in header if column in required]
    assert len(q1_columns) == 257
    assert set(q1_columns) == required

    blank_counts = np.zeros(len(q1_columns), dtype=np.int64)
    spss_counts = np.zeros(len(q1_columns), dtype=np.int64)
    row_count = 0

    for chunk in pd.read_csv(
        wave['path'], usecols=q1_columns, chunksize=RAW_AUDIT_CHUNK_SIZE, low_memory=False
    ):
        chunk = chunk[q1_columns]
        row_count += len(chunk)
        blank_counts += chunk.isna().sum(axis=0).to_numpy(dtype=np.int64)
        spss_counts += chunk.isin(SPSS_MISSING_TOKENS).sum(axis=0).to_numpy(dtype=np.int64)

    total_missing = blank_counts + spss_counts
    assert (total_missing <= row_count).all()
    return pd.DataFrame(
        {
            'wave_index': wave['wave_index'],
            'survey_wave': wave['survey_wave'],
            'source_file': wave['path'].name,
            'variable_position': [header.index(column) + 1 for column in q1_columns],
            'variable': q1_columns,
            'rows_evaluated': row_count,
            'blank_na_count': blank_counts,
            'spss_missing_code_count': spss_counts,
            'total_missing_count': total_missing,
            'observed_count': row_count - total_missing,
            'missing_rate': total_missing / row_count,
            'has_missing': total_missing > 0,
        }
    )


In [9]:
raw_missingness_detail = (
    pd.concat([audit_raw_wave_missingness(wave) for wave in waves], ignore_index=True)
    .sort_values(['wave_index', 'variable_position'])
    .reset_index(drop=True)
)

raw_missingness_summary = raw_missingness_detail.groupby('variable', as_index=False).agg(
    waves_present=('wave_index', 'nunique'),
    rows_evaluated=('rows_evaluated', 'sum'),
    blank_na_count=('blank_na_count', 'sum'),
    spss_missing_code_count=('spss_missing_code_count', 'sum'),
    total_missing_count=('total_missing_count', 'sum'),
    observed_count=('observed_count', 'sum'),
)
raw_missingness_summary['waves_absent'] = len(waves) - raw_missingness_summary['waves_present']
raw_missingness_summary['missing_rate'] = (
    raw_missingness_summary['total_missing_count'] / raw_missingness_summary['rows_evaluated']
)
raw_missingness_summary['has_missing'] = raw_missingness_summary['total_missing_count'].gt(0)
raw_missingness_summary = raw_missingness_summary.sort_values(
    ['missing_rate', 'total_missing_count', 'variable'], ascending=[False, False, True]
).reset_index(drop=True)


wave_missingness_overview = raw_missingness_detail.groupby(
    ['wave_index', 'survey_wave', 'source_file'], as_index=False
).agg(
    rows=('rows_evaluated', 'first'),
    variables=('variable', 'size'),
    variables_with_missing=('has_missing', 'sum'),
    missing_cells=('total_missing_count', 'sum'),
    cells_evaluated=('rows_evaluated', 'sum'),
)
wave_missingness_overview['overall_missing_rate'] = (
    wave_missingness_overview['missing_cells'] / wave_missingness_overview['cells_evaluated']
)

display(wave_missingness_overview.style.format({'overall_missing_rate': '{:.2%}'}))
display(
    raw_missingness_summary.loc[raw_missingness_summary['has_missing']]
    .head(40)
    .style.format({'missing_rate': '{:.2%}'})
)
assert raw_missingness_detail.groupby('wave_index').size().eq(257).all()
assert set(raw_missingness_detail['variable']) <= set(
    Q1_RAW_BASE_COLUMNS + MONTHS_COLUMNS + DAYS_COLUMNS + ['Month', 'month']
)


,wave_index,survey_wave,source_file,rows,variables,variables_with_missing,missing_cells,cells_evaluated,overall_missing_rate
0,1,2015/16,active_lives_1516_london_125.csv,19620,257,108,877043,5042340,17.39%
1,2,2016/17,active_lives_1617_london_125.csv,19248,257,109,835796,4946736,16.90%
2,3,2017/18,2017_data_125_activities.csv,15967,257,107,585797,4103519,14.28%
3,4,2018/19,2018_data_125_activities.csv,15889,257,3,1468,4083473,0.04%
4,5,2019/20,1920_london32_stable125.csv,16091,257,3,1349,4135387,0.03%
5,6,2020/21,2021_london32_stable125.csv,16028,257,109,221408,4119196,5.38%
6,7,2021/22,year7_125activities.csv,16139,257,108,411173,4147723,9.91%
7,8,2022/23,year8_125activities.csv,16515,257,108,392122,4244355,9.24%


,variable,waves_present,rows_evaluated,blank_na_count,spss_missing_code_count,total_missing_count,observed_count,waves_absent,missing_rate,has_missing
0,DAYS10P60GR_AIKIDO_S04,8,135497,23121,8260,31381,104116,0,23.16%,True
1,DAYS10P60GR_AIRGUN_S08,8,135497,23121,8260,31381,104116,0,23.16%,True
2,DAYS10P60GR_BOWLSCARPET_U18,8,135497,23121,8260,31381,104116,0,23.16%,True
3,DAYS10P60GR_BOWLSCROWNGREEN_U19,8,135497,23121,8260,31381,104116,0,23.16%,True
4,DAYS10P60GR_BOWLSFLATGREEN_U20,8,135497,23121,8260,31381,104116,0,23.16%,True
5,DAYS10P60GR_BOWLSSHORTMAT_U23,8,135497,23121,8260,31381,104116,0,23.16%,True
6,DAYS10P60GR_CLIMBWALL_R02,8,135497,23121,8260,31381,104116,0,23.16%,True
7,DAYS10P60GR_CRICKETLONG_Q06,8,135497,23121,8260,31381,104116,0,23.16%,True
8,DAYS10P60GR_CRICKETOTH_Q12,8,135497,23121,8260,31381,104116,0,23.16%,True
9,DAYS10P60GR_CRICKETSHORT_Q07,8,135497,23121,8260,31381,104116,0,23.16%,True


## 4. Respondent cleaning and geography mapping


In [10]:
BASE_COLUMNS = [
    'mode',
    'LA_2023',
    'LondInOut',
    'Age9',
    'Disab3',
    'wt_final',
    'wt_time',
    'MEMS7GR_ALL',
]


def load_clean_wave(wave):
    header = headers[wave['wave_index']]
    month_source = next(column for column in header if column.lower() == 'month')
    usecols = BASE_COLUMNS + [month_source] + MONTHS_COLUMNS + DAYS_COLUMNS
    assert not (set(usecols) - set(header))

    frame = pd.read_csv(wave['path'], usecols=usecols, low_memory=False)
    frame = frame.rename(columns={month_source: 'month_index'})
    numeric_columns = BASE_COLUMNS + ['month_index'] + MONTHS_COLUMNS + DAYS_COLUMNS
    frame[numeric_columns] = frame[numeric_columns].apply(pd.to_numeric, errors='coerce')
    frame = frame.replace(SPSS_MISSING, np.nan)

    eligible = (
        frame['MEMS7GR_ALL'].isin([0, 1, 2])
        & frame['LondInOut'].isin([1, 2])
        & frame['LA_2023'].notna()
        & frame['LA_2023'].ne(CITY_OF_LONDON_LA2023)
    )
    frame = frame.loc[
        eligible, BASE_COLUMNS + ['month_index'] + MONTHS_COLUMNS + DAYS_COLUMNS
    ].copy()
    frame.insert(0, 'survey_wave', wave['survey_wave'])
    frame.insert(0, 'year', wave['wave_index'])
    return frame


respondents = pd.concat([load_clean_wave(wave) for wave in waves], ignore_index=True)

lookup = pd.read_csv(LOOKUP_PATH, low_memory=False)
lookup['Code'] = pd.to_numeric(lookup['Code'], errors='coerce')
la_lookup = (
    lookup[lookup['year'].eq(8) & lookup['Variable'].eq('LA_2023')][['Code', 'CodeLabel']]
    .dropna()
    .drop_duplicates('Code')
)
la_lookup['gss_code'] = la_lookup['CodeLabel'].str.split().str[0]
la_lookup['borough'] = la_lookup['CodeLabel'].str.split(n=1).str[1]
respondents['borough'] = respondents['LA_2023'].map(la_lookup.set_index('Code')['borough'])
respondents['gss_code'] = respondents['LA_2023'].map(la_lookup.set_index('Code')['gss_code'])
respondents['inner_outer'] = respondents['LondInOut'].map(AREA_LABELS)
assert respondents[['borough', 'gss_code', 'inner_outer']].notna().all().all()

annual_respondents = respondents.loc[respondents['wt_final'].gt(0)].copy()
monthly_respondents = respondents.loc[respondents['wt_time'].gt(0)].copy()
assert len(annual_respondents) == 135497
assert len(monthly_respondents) == 134916
assert set(monthly_respondents['month_index'].astype(int)) == set(range(1, 97))
assert annual_respondents['borough'].nunique() == monthly_respondents['borough'].nunique() == 32


## 5. Shared weighted survey aggregation


In [11]:
def kish_effective_n(weights):
    weights = np.asarray(weights, dtype=float)
    return float(weights.sum() ** 2 / np.square(weights).sum())


def weighted_binary_share(values, weights, valid_codes, positive_codes):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isin(values, valid_codes) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.nan
    return float(np.average(np.isin(values[valid], positive_codes), weights=weights[valid]))


def weighted_rate_vector(values, weights, valid_codes, positive_rule):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)[:, None]
    valid = np.isin(values, valid_codes)
    denominator = np.sum(valid * weights, axis=0)
    numerator = np.sum((positive_rule(values) & valid) * weights, axis=0)
    return np.divide(
        numerator,
        denominator,
        out=np.full(values.shape[1], np.nan, dtype=float),
        where=denominator > 0,
    )


def summarise_group(group, weight_column):
    weights = group[weight_column].to_numpy(float)
    total = weights.sum()
    row = {
        'effective_n': kish_effective_n(weights),
        'inactive_rate': float(weights[group['MEMS7GR_ALL'].eq(0).to_numpy()].sum() / total),
        'fairly_active_rate': float(weights[group['MEMS7GR_ALL'].eq(1).to_numpy()].sum() / total),
        'active_rate': float(weights[group['MEMS7GR_ALL'].eq(2).to_numpy()].sum() / total),
        'older_adult_rate': weighted_binary_share(group['Age9'], weights, range(2, 10), [7, 8, 9]),
        'limiting_disability_rate': weighted_binary_share(
            group['Disab3'], weights, [1, 2, 3], [1]
        ),
        'online_response_rate': weighted_binary_share(group['mode'], weights, [1, 2], [1]),
    }
    months12_rates = weighted_rate_vector(
        group[MONTHS_COLUMNS], weights, [0, 1], lambda values: values == 1
    )
    days_any_rates = weighted_rate_vector(
        group[DAYS_COLUMNS], weights, [0, 1, 2], lambda values: values >= 1
    )
    days_2plus_rates = weighted_rate_vector(
        group[DAYS_COLUMNS], weights, [0, 1, 2], lambda values: values == 2
    )
    row.update(dict(zip(MONTHS12_RATE_COLUMNS, months12_rates)))
    row.update(dict(zip(DAYS_ANY_RATE_COLUMNS, days_any_rates)))
    row.update(dict(zip(DAYS_2PLUS_RATE_COLUMNS, days_2plus_rates)))
    return row


def aggregate_panel(frame, group_columns, weight_column):
    rows = []
    grouper = group_columns[0] if len(group_columns) == 1 else group_columns
    for keys, group in frame.groupby(grouper, observed=True, sort=True):
        keys = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_columns, keys))
        row.update(summarise_group(group, weight_column))
        rows.append(row)
    return pd.DataFrame(rows)


## 6. Annual panels and annual external context


In [12]:
london_annual = aggregate_panel(annual_respondents, ['year', 'survey_wave'], 'wt_final')
london_annual['geography_level'] = 'London'
london_annual['geography_code'] = 'LONDON_32'
london_annual['geography_name'] = 'London excluding City of London'

area_annual = aggregate_panel(
    annual_respondents, ['year', 'survey_wave', 'LondInOut'], 'wt_final'
)
area_annual['inner_outer'] = area_annual['LondInOut'].map(AREA_LABELS)
area_annual['geography_level'] = 'InnerOuter'
area_annual['geography_code'] = area_annual['LondInOut'].map(AREA_CODES)
area_annual['geography_name'] = area_annual['inner_outer']

borough_annual = aggregate_panel(
    annual_respondents, ['year', 'survey_wave', 'LA_2023'], 'wt_final'
).merge(
    la_lookup[['Code', 'gss_code', 'borough']],
    left_on='LA_2023', right_on='Code', validate='many_to_one'
)
borough_annual['inner_outer'] = borough_annual['LA_2023'].map(
    annual_respondents.drop_duplicates('LA_2023').set_index('LA_2023')['inner_outer']
)
borough_annual['geography_level'] = 'Borough'
borough_annual['geography_code'] = borough_annual['gss_code']
borough_annual['geography_name'] = borough_annual['borough']

borough_reference = borough_annual[
    ['geography_code', 'geography_name', 'inner_outer']
].drop_duplicates()
borough_external_geo = borough_external_year.merge(
    borough_reference, on=['geography_code', 'geography_name'], validate='many_to_one'
)
crime_annual_counts = crime_counts.groupby(
    ['BoroughName', 'year', 'survey_wave'], as_index=False
)[CRIME_COUNT_COLUMNS].sum()
borough_annual_context = crime_annual_counts.merge(
    borough_external_geo.assign(year=lambda frame: frame['calendar_year'] - 2015),
    left_on=['BoroughName', 'year'],
    right_on=['geography_name', 'year'],
    validate='one_to_one',
)
borough_annual_context['context_reference_year'] = borough_annual_context['calendar_year']
borough_annual_context = add_context_rates(borough_annual_context)
area_annual_context = aggregate_context(
    borough_annual_context, ['year', 'survey_wave', 'inner_outer']
)
area_annual_context['context_reference_year'] = area_annual_context['year'] + 2015
london_annual_context = aggregate_context(borough_annual_context, ['year', 'survey_wave'])
london_annual_context['context_reference_year'] = london_annual_context['year'] + 2015

london_annual = london_annual.merge(
    compact_context(london_annual_context), on=['year', 'survey_wave'], validate='one_to_one'
)
area_annual = area_annual.merge(
    compact_context(area_annual_context),
    on=['year', 'survey_wave', 'inner_outer'],
    validate='one_to_one',
)
borough_annual = borough_annual.merge(
    compact_context(borough_annual_context),
    on=['year', 'survey_wave', 'geography_code', 'geography_name', 'inner_outer'],
    validate='one_to_one',
)


## 7. Monthly and quarterly panels at all three geographic levels

Monthly and quarterly estimates are produced separately for London, Inner/Outer London and all 32
boroughs. Quarterly estimates are calculated directly from respondent records with `wt_time`; they
are not averages of monthly rates. This gives the forecasting notebook nine matched tasks:
three geographic levels × annual, quarterly and monthly frequency.


In [13]:
borough_month_context = crime_counts.merge(
    borough_external_geo,
    left_on=['BoroughName', 'calendar_year'],
    right_on=['geography_name', 'calendar_year'],
    validate='many_to_one',
)
borough_month_context['context_reference_year'] = borough_month_context['calendar_year']
borough_month_context = add_context_rates(borough_month_context)
area_month_context = aggregate_context(
    borough_month_context, ['month_index', 'period_start', 'calendar_year', 'inner_outer']
)
area_month_context['context_reference_year'] = area_month_context['calendar_year']
london_month_context = aggregate_context(
    borough_month_context, ['month_index', 'period_start', 'calendar_year']
)
london_month_context['context_reference_year'] = london_month_context['calendar_year']

london_estimates = aggregate_panel(monthly_respondents, ['month_index'], 'wt_time')
london_monthly = time_reference.merge(london_estimates, on='month_index', how='left', validate='one_to_one')
london_monthly['geography_level'] = 'London'
london_monthly['geography_code'] = 'LONDON_32'
london_monthly['geography_name'] = 'London excluding City of London'
london_monthly = london_monthly.merge(
    compact_context(london_month_context),
    on=['month_index', 'period_start', 'calendar_year'], validate='one_to_one',
)

area_reference = pd.DataFrame({
    'LondInOut': [1, 2],
    'inner_outer': [AREA_LABELS[1], AREA_LABELS[2]],
    'geography_code': [AREA_CODES[1], AREA_CODES[2]],
})
area_estimates = aggregate_panel(monthly_respondents, ['month_index', 'LondInOut'], 'wt_time')
area_monthly = time_reference.merge(area_reference, how='cross').merge(
    area_estimates, on=['month_index', 'LondInOut'], how='left', validate='one_to_one'
)
area_monthly['geography_level'] = 'InnerOuter'
area_monthly['geography_name'] = area_monthly['inner_outer']
area_monthly = area_monthly.merge(
    compact_context(area_month_context),
    on=['month_index', 'period_start', 'calendar_year', 'inner_outer'], validate='one_to_one',
)

borough_estimates = aggregate_panel(monthly_respondents, ['month_index', 'LA_2023'], 'wt_time')
borough_monthly = time_reference.merge(
    annual_respondents[['LA_2023', 'gss_code', 'borough', 'inner_outer']].drop_duplicates(),
    how='cross',
).merge(
    borough_estimates, on=['month_index', 'LA_2023'], how='left', validate='one_to_one'
)
borough_monthly['geography_level'] = 'Borough'
borough_monthly['geography_code'] = borough_monthly['gss_code']
borough_monthly['geography_name'] = borough_monthly['borough']
borough_monthly = borough_monthly.merge(
    compact_context(borough_month_context).drop(
        columns=['year', 'survey_wave', 'quarter_index'], errors='ignore'
    ),
    on=['month_index', 'period_start', 'calendar_year',
        'geography_code', 'geography_name', 'inner_outer'],
    validate='one_to_one',
)

monthly_respondents['quarter_index'] = ((monthly_respondents['month_index'] - 1) // 3 + 1).astype(int)
quarter_reference = time_reference.groupby('quarter_index', as_index=False).agg(
    year=('year', 'first'), survey_wave=('survey_wave', 'first'),
    quarter_of_wave=('month_of_wave', lambda values: int((values.min() - 1) // 3 + 1)),
    quarter_start=('period_start', 'min'), quarter_end=('period_start', 'max'),
)

quarter_context_rows = []
for (borough, quarter_index), group in borough_month_context.groupby(
    ['geography_name', 'quarter_index'], sort=True
):
    row = {
        'geography_name': borough, 'quarter_index': quarter_index,
        'geography_code': group['geography_code'].iloc[0],
        'inner_outer': group['inner_outer'].iloc[0],
        'context_reference_year': group.sort_values('period_start')['calendar_year'].iloc[-1],
        'total_population': group['total_population'].mean(),
        'adult_population': group['adult_population'].mean(),
        'age_16_34_population': group['age_16_34_population'].mean(),
        'age_65_plus_population': group['age_65_plus_population'].mean(),
        'female_adult_population': group['female_adult_population'].mean(),
        'net_migration': group['net_migration'].mean(),
        'median_weekly_earnings_gbp': np.average(
            group['median_weekly_earnings_gbp'], weights=group['adult_population']
        ),
        'earnings_relative_to_london': np.average(
            group['earnings_relative_to_london'], weights=group['adult_population']
        ),
        'earnings_confidence_pct': np.average(
            group['earnings_confidence_pct'], weights=group['adult_population']
        ),
    }
    for count_column in CRIME_COUNT_COLUMNS:
        row[count_column] = group[count_column].sum()
    quarter_context_rows.append(row)
borough_quarter_context = add_context_rates(pd.DataFrame(quarter_context_rows))
area_quarter_context = aggregate_context(
    borough_quarter_context, ['quarter_index', 'inner_outer']
).merge(
    quarter_reference[['quarter_index', 'quarter_end']], on='quarter_index', validate='many_to_one'
)
area_quarter_context['context_reference_year'] = area_quarter_context['quarter_end'].dt.year
london_quarter_context = aggregate_context(borough_quarter_context, ['quarter_index']).merge(
    quarter_reference[['quarter_index', 'quarter_end']], on='quarter_index', validate='one_to_one'
)
london_quarter_context['context_reference_year'] = london_quarter_context['quarter_end'].dt.year

london_quarter_estimates = aggregate_panel(monthly_respondents, ['quarter_index'], 'wt_time')
london_quarterly = quarter_reference.merge(
    london_quarter_estimates, on='quarter_index', how='left', validate='one_to_one'
)
london_quarterly['geography_level'] = 'London'
london_quarterly['geography_code'] = 'LONDON_32'
london_quarterly['geography_name'] = 'London excluding City of London'
london_quarterly = london_quarterly.merge(
    compact_context(london_quarter_context).drop(columns='quarter_end', errors='ignore'),
    on='quarter_index', validate='one_to_one',
)

area_quarter_estimates = aggregate_panel(
    monthly_respondents, ['quarter_index', 'LondInOut'], 'wt_time'
)
area_quarterly = quarter_reference.merge(area_reference, how='cross').merge(
    area_quarter_estimates, on=['quarter_index', 'LondInOut'], how='left', validate='one_to_one'
)
area_quarterly['geography_level'] = 'InnerOuter'
area_quarterly['geography_name'] = area_quarterly['inner_outer']
area_quarterly = area_quarterly.merge(
    compact_context(area_quarter_context).drop(columns='quarter_end', errors='ignore'),
    on=['quarter_index', 'inner_outer'], validate='one_to_one',
)

borough_quarter_estimates = aggregate_panel(
    monthly_respondents, ['quarter_index', 'LA_2023'], 'wt_time'
)
borough_quarterly = quarter_reference.merge(
    annual_respondents[['LA_2023', 'gss_code', 'borough', 'inner_outer']].drop_duplicates(),
    how='cross',
).merge(
    borough_quarter_estimates, on=['quarter_index', 'LA_2023'], how='left', validate='one_to_one'
)
borough_quarterly['geography_level'] = 'Borough'
borough_quarterly['geography_code'] = borough_quarterly['gss_code']
borough_quarterly['geography_name'] = borough_quarterly['borough']
borough_quarterly = borough_quarterly.merge(
    compact_context(borough_quarter_context),
    on=['quarter_index', 'geography_code', 'geography_name', 'inner_outer'],
    validate='one_to_one',
)


In [14]:
TIME_ORDER = [
    'month_index', 'quarter_index', 'year', 'survey_wave', 'month_of_wave',
    'period_start', 'calendar_year',
]
QUARTER_TIME_ORDER = [
    'quarter_index', 'year', 'survey_wave', 'quarter_of_wave', 'quarter_start', 'quarter_end',
]
GEO_ORDER = ['geography_level', 'geography_code', 'geography_name']
MEASURE_ORDER = (
    QUALITY_COLUMNS + OUTCOME_COLUMNS + COMPOSITION_FEATURES + ACTIVITY_FEATURES
    + EXTERNAL_CONTEXT_COLUMNS + EXTERNAL_QUALITY_COLUMNS + ['context_reference_year']
)
ANNUAL_ORDER = ['year', 'survey_wave'] + GEO_ORDER
london_annual = london_annual[ANNUAL_ORDER + MEASURE_ORDER]
area_annual = area_annual[ANNUAL_ORDER + ['inner_outer'] + MEASURE_ORDER]
borough_annual = borough_annual[ANNUAL_ORDER + ['inner_outer'] + MEASURE_ORDER]
london_monthly = london_monthly[TIME_ORDER + GEO_ORDER + MEASURE_ORDER]
area_monthly = area_monthly[TIME_ORDER + GEO_ORDER + ['inner_outer'] + MEASURE_ORDER]
borough_monthly = borough_monthly[TIME_ORDER + GEO_ORDER + ['inner_outer'] + MEASURE_ORDER]
london_quarterly = london_quarterly[QUARTER_TIME_ORDER + GEO_ORDER + MEASURE_ORDER]
area_quarterly = area_quarterly[QUARTER_TIME_ORDER + GEO_ORDER + ['inner_outer'] + MEASURE_ORDER]
borough_quarterly = borough_quarterly[QUARTER_TIME_ORDER + GEO_ORDER + ['inner_outer'] + MEASURE_ORDER]


## 8. Age-specific annual evidence for decomposition

Four consistent adult age groups are prepared at London, Inner/Outer and borough level. Survey
activity rates use `wt_final`; ONS age shares use the matching calendar year. These compact tables
are stored as sheets in the audit workbook because they support explanation rather than forecasting.


In [15]:
AGE_BANDS = ['16-34', '35-54', '55-64', '65+']
AGE_CODE_TO_BAND = {
    2: '16-34', 3: '16-34', 4: '35-54', 5: '35-54',
    6: '55-64', 7: '65+', 8: '65+', 9: '65+',
}
age_respondents = annual_respondents.loc[
    annual_respondents['Age9'].isin(AGE_CODE_TO_BAND)
].copy()
age_respondents['age_band'] = age_respondents['Age9'].map(AGE_CODE_TO_BAND)
age_respondents['calendar_year'] = 2015 + age_respondents['year']


def summarise_age_group(group):
    weights = group['wt_final'].to_numpy(float)
    total = weights.sum()
    row = {'respondent_n': len(group), 'effective_n': total**2 / np.square(weights).sum()}
    for code_value, target in zip([0, 1, 2], OUTCOME_COLUMNS):
        row[target] = weights[group['MEMS7GR_ALL'].eq(code_value).to_numpy()].sum() / total
    return pd.Series(row)


population_age = population_long.loc[population_long['age'].ge(16)].copy()
population_age['age_band'] = pd.cut(
    population_age['age'], bins=[15, 34, 54, 64, np.inf], labels=AGE_BANDS, right=True
)
population_age = population_age.groupby(
    ['ladcode23', 'laname23', 'calendar_year', 'age_band'], observed=True, as_index=False
)['population'].sum()
population_age['adult_population'] = population_age.groupby(
    ['ladcode23', 'calendar_year']
)['population'].transform('sum')
population_age['population_share'] = population_age['population'] / population_age['adult_population']

age_borough_annual = age_respondents.groupby(
    ['year', 'survey_wave', 'calendar_year', 'gss_code', 'borough', 'inner_outer', 'age_band'],
    observed=True, sort=True,
).apply(summarise_age_group, include_groups=False).reset_index().rename(
    columns={'gss_code': 'geography_code', 'borough': 'geography_name'}
)
age_borough_annual = age_borough_annual.merge(
    population_age[['ladcode23', 'calendar_year', 'age_band', 'population', 'population_share']],
    left_on=['geography_code', 'calendar_year', 'age_band'],
    right_on=['ladcode23', 'calendar_year', 'age_band'], validate='one_to_one',
).drop(columns='ladcode23')


def aggregate_age_level(group_columns, geography_level):
    survey = age_respondents.groupby(group_columns + ['age_band'], observed=True, sort=True).apply(
        summarise_age_group, include_groups=False
    ).reset_index()
    population_keys = [column for column in group_columns if column in ['year', 'calendar_year', 'inner_outer']]
    population_base = age_borough_annual.copy()
    population = population_base.groupby(population_keys + ['age_band'], observed=True, as_index=False)[
        'population'
    ].sum()
    population['population_share'] = population['population'] / population.groupby(population_keys)[
        'population'
    ].transform('sum')
    result = survey.merge(population, on=population_keys + ['age_band'], validate='one_to_one')
    result['geography_level'] = geography_level
    return result


age_london_annual = aggregate_age_level(
    ['year', 'survey_wave', 'calendar_year'], 'London'
)
age_london_annual['geography_code'] = 'LONDON_32'
age_london_annual['geography_name'] = 'London excluding City of London'
age_area_annual = aggregate_age_level(
    ['year', 'survey_wave', 'calendar_year', 'inner_outer'], 'InnerOuter'
)
age_area_annual['geography_code'] = age_area_annual['inner_outer'].map(
    {'Inner London': AREA_CODES[1], 'Outer London': AREA_CODES[2]}
)
age_area_annual['geography_name'] = age_area_annual['inner_outer']

assert len(age_london_annual) == 8 * 4
assert len(age_area_annual) == 2 * 8 * 4
assert len(age_borough_annual) == 32 * 8 * 4


## 9. Validation and compact delivery

Nine forecasting panels are the only CSV outputs. Missingness, source decisions, age-specific
tables, panel dimensions and the variable dictionary are consolidated into one Excel workbook.


In [16]:
def validate_panel(panel, keys, expected_rows, expected_geographies, missing_rows=0):
    assert len(panel) == expected_rows
    assert not panel.duplicated(keys).any()
    assert panel['geography_code'].nunique() == expected_geographies
    observed_mask = panel[OUTCOME_COLUMNS].notna().all(axis=1)
    assert int((~observed_mask).sum()) == missing_rows
    observed = panel.loc[observed_mask]
    assert observed[OUTCOME_COLUMNS + COMPOSITION_FEATURES].notna().all().all()
    assert observed[OUTCOME_COLUMNS + COMPOSITION_FEATURES].apply(
        lambda column: column.between(0, 1)
    ).all().all()
    assert np.allclose(observed[OUTCOME_COLUMNS].sum(axis=1), 1, atol=1e-10)
    assert observed['effective_n'].gt(0).all()
    assert observed[ACTIVITY_FEATURES].stack().between(0, 1).all()
    assert panel[EXTERNAL_CONTEXT_COLUMNS + EXTERNAL_QUALITY_COLUMNS].notna().all().all()


panels = {
    'london_annual': london_annual, 'area_annual': area_annual, 'borough_annual': borough_annual,
    'london_quarterly': london_quarterly, 'area_quarterly': area_quarterly,
    'borough_quarterly': borough_quarterly,
    'london_monthly': london_monthly, 'area_monthly': area_monthly,
    'borough_monthly': borough_monthly,
}
expectations = {
    'london_annual': (['year'], 8, 1, 0),
    'area_annual': (['year', 'geography_code'], 16, 2, 0),
    'borough_annual': (['year', 'geography_code'], 256, 32, 0),
    'london_quarterly': (['quarter_index'], 32, 1, 0),
    'area_quarterly': (['quarter_index', 'geography_code'], 64, 2, 0),
    'borough_quarterly': (['quarter_index', 'geography_code'], 1024, 32, 0),
    'london_monthly': (['month_index'], 96, 1, 0),
    'area_monthly': (['month_index', 'geography_code'], 192, 2, 0),
    'borough_monthly': (['month_index', 'geography_code'], 3072, 32, 1),
}
for name, panel in panels.items():
    validate_panel(panel, *expectations[name])
    panel.to_csv(OUTPUTS[name], index=False, encoding='utf-8-sig')

dictionary_rows = []
for column in borough_monthly.columns:
    if column in OUTCOME_COLUMNS:
        role, source, units = 'outcome', 'Active Lives / survey-weighted', 'proportion'
    elif column in COMPOSITION_FEATURES:
        role, source, units = 'survey composition/control', 'Active Lives', 'proportion'
    elif column in QUALITY_COLUMNS or column in EXTERNAL_QUALITY_COLUMNS:
        role, source, units = 'quality metadata', 'survey or external source', 'source-specific'
    elif column in DEMOGRAPHIC_EARNINGS_COLUMNS:
        role, source, units = 'population/migration/earnings context', 'ONS / ASHE', 'source-specific'
    elif column in CRIME_RATE_COLUMNS:
        role, source, units = 'selected recorded-crime context', 'Metropolitan Police', 'per 1,000'
    elif column.startswith(('MONTHS12_RATE_', 'DAYS_ANY_RATE_', 'DAYS_2PLUS_RATE_')):
        role, source, units = 'activity predictor', 'Active Lives / survey-weighted', 'proportion'
    else:
        role, source, units = 'identifier', 'constructed', 'identifier'
    dictionary_rows.append({'variable': column, 'role': role, 'source': source, 'units': units})
variable_dictionary = pd.DataFrame(dictionary_rows)

panel_summary = pd.DataFrame([{
    'panel': name, 'file': OUTPUTS[name].name, 'rows': len(panel),
    'columns': len(panel.columns),
    'all_target_missing_rows': int(panel[OUTCOME_COLUMNS].isna().all(axis=1).sum()),
} for name, panel in panels.items()])
external_quality = pd.DataFrame([{
    'variable': column,
    'missing': int(borough_external_year[column].isna().sum()) if column in borough_external_year else 0,
    'minimum': pd.concat([borough_annual[column], borough_monthly[column], borough_quarterly[column]]).min(),
    'maximum': pd.concat([borough_annual[column], borough_monthly[column], borough_quarterly[column]]).max(),
} for column in EXTERNAL_CONTEXT_COLUMNS + EXTERNAL_QUALITY_COLUMNS])

with pd.ExcelWriter(AUDIT_OUTPUT, engine='openpyxl') as writer:
    external_inventory.to_excel(writer, sheet_name='external_sources', index=False)
    panel_summary.to_excel(writer, sheet_name='panel_summary', index=False)
    external_quality.to_excel(writer, sheet_name='external_quality', index=False)
    raw_missingness_detail.to_excel(writer, sheet_name='raw_missing_by_wave', index=False)
    raw_missingness_summary.to_excel(writer, sheet_name='raw_missing_summary', index=False)
    variable_dictionary.to_excel(writer, sheet_name='variable_dictionary', index=False)
    age_london_annual.to_excel(writer, sheet_name='age_london_annual', index=False)
    age_area_annual.to_excel(writer, sheet_name='age_inner_outer_annual', index=False)
    age_borough_annual.to_excel(writer, sheet_name='age_borough_annual', index=False)

display(panel_summary)
display(external_quality)


,panel,file,rows,columns,all_target_missing_rows
0,london_annual,q1_london_annual.csv,8,396,0
1,area_annual,q1_inner_outer_annual.csv,16,397,0
2,borough_annual,q1_borough_annual.csv,256,397,0
3,london_quarterly,q1_london_quarterly.csv,32,400,0
4,area_quarterly,q1_inner_outer_quarterly.csv,64,401,0
5,borough_quarterly,q1_borough_quarterly.csv,1024,401,0
6,london_monthly,q1_london_monthly.csv,96,401,0
7,area_monthly,q1_inner_outer_monthly.csv,192,402,0
8,borough_monthly,q1_borough_monthly.csv,3072,402,1


,variable,missing,minimum,maximum
0,adult_population,0,123715.000000,320227.000000
1,population_age_16_34_rate,0,0.242239,0.547306
2,population_age_65_plus_rate,0,0.069031,0.226077
3,population_female_rate,0,0.492150,0.542308
4,net_migration_per_1000,0,-26.059914,27.591641
5,median_weekly_earnings_gbp,0,421.600000,855.900000
6,earnings_relative_to_london,0,0.792805,1.402007
7,personal_safety_crime_rate_per_1000,0,1.008696,78.214226
8,property_public_space_crime_rate_per_1000,0,1.111086,286.179479
9,selected_recorded_crime_rate_per_1000,0,2.603988,364.393706


### Delivery

The data layer now contains nine matched forecasting panels: London, Inner/Outer London and 32
boroughs at annual, quarterly and monthly frequency. The audit workbook also contains the
age-specific evidence used for population decomposition. Only population/migration, resident
earnings and selected recorded crime are retained as external sources.
